In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn

from load_data import  CatDogDataLoadandSave
from vgg16_model import VGG16

### Load data and Save dataset as ubyte format


In [ ]:
data_path = r'/Users/gimoon/Documents/GitHub/Data'
train_data_path = os.path.join(data_path, "cat-and-dog", "training_set")
test_data_path  = os.path.join(data_path, "cat-and-dog", "test_set")



In [15]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    transforms.ToTensor(),
])

train_data = datasets.ImageFolder(train_data_path, transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)#, pin_memory=True)

In [16]:
train_data

Dataset ImageFolder
    Number of datapoints: 8005
    Root location: /Users/gimoon/Documents/GitHub/Data/cat-and-dog/training_set
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
               ToTensor()
           )

### create torch data loader

In [ ]:
dataset_train = CatandDogDataLoader(raw_folder=train_data_path, train=True)
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)

### Build VGG16 architecture

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VGG16(3, 2).to(device)

device

In [ ]:
from torchsummary import summary
summary(model, (3, 224, 224))


In [ ]:
learning_rate = 1e-4
num_epochs = 20

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
model.train()

for epoch in range(num_epochs):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    for batch_idx, (inputs, labels) in ProgressBar:

        # Ensure labels are torch.long before moving to device for CrossEntropyLoss
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()
        # print(inputs.dtype, labels.dtype)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())